# Step 1 — Preprocessing and ResNet50 + VGG16 Feature Extraction

Code for the article *Transfer Learning for Multi-Label Multi-Class Classification of Cervical Spine Disorders on Magnetic Resonance Imaging* (Rehman, Farhan & Siddiqui).

This notebook:
1. loads the image manifest (`CervicalSpine/processed/manifest.csv`: 1,464 sagittal MRI slices from 61 de-identified patients, multi-label disease columns);
2. preprocesses every slice (1st–99th percentile intensity windowing, foreground crop with 8% margin, median + bilateral denoising, square padding, 224 × 224 resize, grayscale → RGB) and caches it as JPEG;
3. extracts frozen ImageNet ResNet50 (2,048-d) and VGG16 (512-d) global-average-pooled features and concatenates them into a 2,560-d vector per image.

**Outputs** (written to `MyDrive/CervicalSpine/dual_eval_output/`):
- `prep_cache/cached_manifest.csv` and `prep_cache/images/`
- `feature_cache/concat_<hash>.npz` (1,464 × 2,560)

**Before running:** place the dataset so that the images are at `MyDrive/CervicalSpine/processed/<patient_id>/<patient_id>_slNNN.jpg` and the manifest at `MyDrive/CervicalSpine/processed/manifest.csv`. Use a GPU runtime (Runtime → Change runtime type → T4 GPU), then **Runtime → Run all**.

Next step: `02_classification_and_results.ipynb`.

## 0 — Computing environment (for the record)

In [ ]:
# Records the computing environment used for this run (hardware + library versions).
import platform, sys, subprocess
print("Python      :", sys.version.split()[0])
print("Platform    :", platform.platform())
print("CPU cores   :", subprocess.run(["nproc"], capture_output=True, text=True).stdout.strip())
print(subprocess.run(["bash", "-c", "lscpu | grep 'Model name'"], capture_output=True, text=True).stdout.strip())
print(subprocess.run(["free", "-h"], capture_output=True, text=True).stdout)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)
import importlib
for lib in ["tensorflow", "numpy", "pandas", "sklearn", "cv2", "matplotlib", "PIL", "openpyxl"]:
    try:
        m = importlib.import_module(lib)
        print(f"{lib:12s}:", getattr(m, "__version__", "?"))
    except ImportError:
        print(f"{lib:12s}: not installed")

## 1 — Mount Google Drive, imports, random seed

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install scikit-learn matplotlib opencv-python-headless openpyxl pillow

import os, gc, json, math, random, warnings, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50, VGG16
from tensorflow.keras.applications.resnet50 import preprocess_input as rpre
from tensorflow.keras.applications.vgg16    import preprocess_input as vpre

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TensorFlow:', tf.__version__)
print('GPU       :', tf.config.list_physical_devices('GPU'))

Mounted at /content/drive
TensorFlow: 2.20.0
GPU       : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2 — Configuration

In [ ]:
CONFIG = {
    # --- EDIT THESE ---------------------------------------------------------
    'MANIFEST_CANDIDATES': [
        '/content/drive/MyDrive/CervicalSpine/processed/manifest.csv',
        '/content/drive/MyDrive/CervicalSpine/local_cervical_clean_dataset/clean_manifest.csv',
        '/content/drive/MyDrive/CervicalSpine/article_dataset/curated_manifest.csv',
    ],
    'OUT_DIR': '/content/drive/MyDrive/CervicalSpine/dual_eval_output',

    # --- preprocessing ------------------------------------------------------
    'IMG_SIZE'            : 224,
    'SLICES_PER_PATIENT'  : None,           # None = use every slice; or e.g. 32
    'DENOISE_METHOD'      : 'median_bilateral',
    'AUTO_FOREGROUND_CROP': True,
    'REBUILD_PREP_CACHE'  : False,

    # --- splits -------------------------------------------------------------
    'TEST_SIZE'           : 0.20,
    'VAL_SIZE_FROM_TRAIN' : 0.15,

    # --- diseases (auto-skipped if absent / too few) ------------------------
    'ALL_DISEASES': [
        'Disc Buldging', 'Myleopathy', 'Cord Edma', 'Disc Herniation',
        'Stenosis', 'Scoliotic Deformity', 'Osteophytosis', 'Spondylodiscitis',
    ],
}
Path(CONFIG['OUT_DIR']).mkdir(parents=True, exist_ok=True)
print('OUT_DIR :', CONFIG['OUT_DIR'])
print('Diseases:', CONFIG['ALL_DISEASES'])

OUT_DIR : /content/drive/MyDrive/CervicalSpine/dual_eval_output
Diseases: ['Disc Buldging', 'Myleopathy', 'Cord Edma', 'Disc Herniation', 'Stenosis', 'Scoliotic Deformity', 'Osteophytosis', 'Spondylodiscitis']


## 3 — Load manifest and preprocess images
Each slice is preprocessed once and cached to Drive. The cache is reused on later runs (set `REBUILD_PREP_CACHE = True` in the configuration to rebuild it).

In [ ]:
# ---- column constants & helpers ----
NON_LABEL_COLS = {
    'path','prep_path','patient_id','is_normal','split','disease_count','diseases',
    'source_dataset','source_patient_id','external_cervical_spondylosis','modality',
    'original_nii','source_slice_index','quality_score','class_name','label',
    'folder','ID','Name','Normal','is_augmented',
}
ALIASES = {
    'disc bulging':'Disc Buldging','disc buldging':'Disc Buldging',
    'disc_bulging':'Disc Buldging','disc_buldging':'Disc Buldging',
    'myelopathy':'Myleopathy','myleopathy':'Myleopathy',
    'cord edema':'Cord Edma','cord edma':'Cord Edma',
    'disc herniation':'Disc Herniation',
}

def norm_pid(x):
    s = str(x).strip()
    if s.endswith('.0'): s = s[:-2]
    return s.zfill(3) if s.isdigit() else s

def coerce_binary(v):
    if pd.isna(v): return 0
    if isinstance(v, str):
        t = v.strip().lower()
        if t in {'1','yes','y','true','positive','present'}: return 1
        if t in {'0','no','n','false','negative','absent',''}: return 0
    try: return 1 if float(v) > 0 else 0
    except: return 0

def standardize_columns(df):
    ren = {}
    for c in df.columns:
        k = str(c).strip().lower().replace('_',' ')
        if k in ALIASES and c != ALIASES[k]: ren[c] = ALIASES[k]
    return df.rename(columns=ren) if ren else df

def detect_disease_cols(df):
    cols = []
    for c in df.columns:
        if c in NON_LABEL_COLS or str(c).startswith('MMCSD_'): continue
        vals = pd.to_numeric(df[c], errors='coerce').dropna().unique()
        if len(vals) and set(vals).issubset({0,1}): cols.append(c)
    pref = CONFIG['ALL_DISEASES']
    return [c for c in pref if c in cols] + [c for c in cols if c not in pref]

def choose_manifest(cands):
    for p in cands:
        if Path(p).exists(): return Path(p)
    raise FileNotFoundError(
        'No manifest CSV found. Edit CONFIG["MANIFEST_CANDIDATES"] to point at your file.\n'
        'It must have columns: path, patient_id, and one 0/1 column per disease.\nChecked:\n  ' +
        '\n  '.join(cands))

def foreground_bbox(img, margin_frac=0.08):
    thr = max(5, int(np.percentile(img, 12)))
    ys, xs = np.where(img > thr)
    if not len(xs): return 0, 0, img.shape[1], img.shape[0]
    x1,x2 = xs.min(), xs.max()+1; y1,y2 = ys.min(), ys.max()+1
    w,h = x2-x1, y2-y1; mx,my = int(w*margin_frac), int(h*margin_frac)
    return max(0,x1-mx), max(0,y1-my), min(img.shape[1],x2+mx), min(img.shape[0],y2+my)

def pad_square(img):
    h,w = img.shape[:2]; s = max(h,w)
    t=(s-h)//2; b=s-h-t; l=(s-w)//2; r=s-w-l
    return cv2.copyMakeBorder(img, t,b,l,r, cv2.BORDER_CONSTANT, value=0)

def _denoise(img):
    m = CONFIG['DENOISE_METHOD']
    if m == 'gaussian': return cv2.GaussianBlur(img,(3,3),0)
    if m == 'median': return cv2.medianBlur(img,3)
    if m == 'bilateral': return cv2.bilateralFilter(img,5,35,35)
    if m == 'median_bilateral': return cv2.bilateralFilter(cv2.medianBlur(img,3),5,35,35)
    return img

def preprocess_one(src, dst):
    img = cv2.imread(str(src), cv2.IMREAD_GRAYSCALE)
    if img is None or img.size == 0: return False
    lo, hi = np.percentile(img, [1,99])
    if hi > lo:
        img = np.clip((img.astype(np.float32)-lo)/(hi-lo), 0, 1)
        img = (img*255).astype(np.uint8)
    if CONFIG['AUTO_FOREGROUND_CROP']:
        x1,y1,x2,y2 = foreground_bbox(img); img = img[y1:y2, x1:x2]
    img = _denoise(img); img = pad_square(img)
    img = cv2.resize(img, (CONFIG['IMG_SIZE'],CONFIG['IMG_SIZE']), interpolation=cv2.INTER_AREA)
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    Path(dst).parent.mkdir(parents=True, exist_ok=True)
    return bool(cv2.imwrite(str(dst), img, [int(cv2.IMWRITE_JPEG_QUALITY),95]))

def select_slices(df):
    n = CONFIG['SLICES_PER_PATIENT']
    df = df.sort_values(['patient_id','source_slice_index','path']).copy()
    if n is None: return df
    rows = []
    for _, g in df.groupby('patient_id', sort=True):
        if len(g) > n:
            idx = np.linspace(0, len(g)-1, n).round().astype(int); g = g.iloc[idx]
        rows.append(g)
    return pd.concat(rows, ignore_index=True) if rows else df.iloc[0:0]

# ---- load manifest ----
MANIFEST_PATH = choose_manifest(CONFIG['MANIFEST_CANDIDATES'])
RAW = standardize_columns(pd.read_csv(MANIFEST_PATH))
assert 'path' in RAW.columns and 'patient_id' in RAW.columns, 'manifest needs path + patient_id columns'
RAW['patient_id'] = RAW['patient_id'].map(norm_pid)
RAW['path'] = RAW['path'].astype(str)
RAW = RAW[RAW['path'].map(os.path.exists)].copy()
assert not RAW.empty, 'No existing image paths found in manifest.'

DISEASE_COLS = detect_disease_cols(RAW)
for c in DISEASE_COLS: RAW[c] = RAW[c].map(coerce_binary).astype(int)
if 'is_normal' in RAW.columns:   RAW['is_normal'] = RAW['is_normal'].map(coerce_binary).astype(int)
elif 'Normal' in RAW.columns:    RAW['is_normal'] = RAW['Normal'].map(coerce_binary).astype(int)
else:                            RAW['is_normal'] = (RAW[DISEASE_COLS].sum(axis=1)==0).astype(int)
RAW['disease_count'] = RAW[DISEASE_COLS].sum(axis=1).astype(int)
RAW['source_slice_index'] = pd.to_numeric(RAW.get('source_slice_index', np.nan), errors='coerce')

pf = RAW.groupby('patient_id').first().reset_index()
print('Manifest:', MANIFEST_PATH)
print('Images  :', len(RAW), '| Patients:', RAW.patient_id.nunique())
print(pd.DataFrame([{'disease':'Normal','patients':int(pf['is_normal'].sum())}] +
                   [{'disease':c,'patients':int(pf[c].sum())} for c in DISEASE_COLS]).to_string(index=False))

# ---- preprocess + cache ----
prep_csv = Path(CONFIG['OUT_DIR'])/'prep_cache'/'cached_manifest.csv'
if prep_csv.exists() and not CONFIG['REBUILD_PREP_CACHE']:
    print('\nLoading existing preprocessed manifest...')
    CACHED = pd.read_csv(prep_csv)
    CACHED['patient_id'] = CACHED['patient_id'].map(norm_pid)
    CACHED['prep_path']  = CACHED['prep_path'].astype(str)
    CACHED = CACHED[CACHED['prep_path'].map(os.path.exists)].copy()
else:
    df_s = select_slices(RAW)
    root = Path(CONFIG['OUT_DIR'])/'prep_cache'/'images'
    rows, failed = [], 0
    for i, r in df_s.reset_index(drop=True).iterrows():
        dst = root/str(r['patient_id'])/f"{i:06d}_{Path(r['path']).stem}.jpg"
        ok = dst.exists() and not CONFIG['REBUILD_PREP_CACHE']
        if not ok: ok = preprocess_one(r['path'], dst)
        if ok:
            row = r.to_dict(); row['prep_path'] = str(dst); rows.append(row)
        else: failed += 1
        if (i+1) % 300 == 0: print(f'  {i+1}/{len(df_s)}'); gc.collect()
    CACHED = pd.DataFrame(rows)
    CACHED.to_csv(prep_csv, index=False)
    print(f'Preprocessed: {len(CACHED)} ok | {failed} failed')

for c in DISEASE_COLS:
    if c in CACHED.columns: CACHED[c] = CACHED[c].map(coerce_binary).astype(int)
if 'is_normal' not in CACHED.columns:
    CACHED['is_normal'] = (CACHED[DISEASE_COLS].sum(axis=1)==0).astype(int)
CACHED['is_normal']     = CACHED['is_normal'].map(coerce_binary).astype(int)
CACHED['disease_count'] = CACHED[DISEASE_COLS].sum(axis=1).astype(int)
print('\nCACHED:', len(CACHED), 'images |', CACHED.patient_id.nunique(), 'patients')

Manifest: /content/drive/MyDrive/CervicalSpine/processed/manifest.csv
Images  : 1464 | Patients: 61
            disease  patients
             Normal        18
      Disc Buldging        41
         Myleopathy         4
          Cord Edma         4
    Disc Herniation         4
           Stenosis         1
Scoliotic Deformity         1
      Osteophytosis         1
   Spondylodiscitis         1
  300/1464
  600/1464
  900/1464
  1200/1464
Preprocessed: 1464 ok | 0 failed

CACHED: 1464 images | 61 patients


## 4 — Fused ResNet50 + VGG16 features
Both backbones are frozen (ImageNet weights). Each image is passed through the backbone's own `preprocess_input` and global average pooling, and the two vectors are concatenated (2,048 + 512 = 2,560). Features are cached so extraction runs only once.

In [ ]:
IMG = CONFIG['IMG_SIZE']
OUT = Path(CONFIG['OUT_DIR'])
FEAT_DIR = OUT/'feature_cache'; FEAT_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_TAG = 'resnet50+vgg16|gap|shared_v1'

def _extractor(backbone):
    pre  = {'resnet50': rpre, 'vgg16': vpre}[backbone]
    Base = {'resnet50': ResNet50, 'vgg16': VGG16}[backbone]
    inp  = keras.Input((IMG, IMG, 3))
    x    = layers.Lambda(pre)(inp)
    base = Base(include_top=False, weights='imagenet', input_shape=(IMG, IMG, 3))
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base(x, training=False))
    return keras.Model(inp, x)

def _img_ds(paths, tta=None):
    ds = tf.data.Dataset.from_tensor_slices(np.array(paths, dtype=str))
    def load(p):
        im = tf.io.read_file(p); im = tf.image.decode_jpeg(im, channels=3)
        im = tf.image.resize(im, (IMG, IMG)); im = tf.cast(im, tf.float32)
        if   tta == 'bright_up': im = tf.clip_by_value(im * 1.10, 0, 255)
        elif tta == 'bright_dn': im = tf.clip_by_value(im * 0.90, 0, 255)
        elif tta == 'contrast':  im = tf.clip_by_value((im - 127.5) * 1.12 + 127.5, 0, 255)
        elif tta == 'zoom':      im = tf.image.resize(tf.image.central_crop(im, 0.88), (IMG, IMG))
        return im
    return ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(64).prefetch(tf.data.AUTOTUNE)

def _concat_once(paths, tta=None):
    parts = []
    for bb in ['resnet50', 'vgg16']:
        m = _extractor(bb)
        parts.append(m.predict(_img_ds(paths, tta), verbose=0).astype(np.float32))
        del m; tf.keras.backend.clear_session(); gc.collect()
    return np.concatenate(parts, axis=1)

def concat_features(df):
    paths = df['prep_path'].astype(str).tolist()
    sig   = '|'.join(sorted(paths)) + f'|{IMG}|{EXTRACT_TAG}'
    key   = hashlib.sha1(sig.encode()).hexdigest()[:12]
    fp    = FEAT_DIR/f'concat_{key}.npz'
    if fp.exists():
        d = np.load(fp, allow_pickle=True)
        print('  loaded cached concat features', d['feats'].shape); return d['feats'], {p:i for i,p in enumerate(d['paths'].astype(str))}
    print(f'  extracting concat(ResNet50+VGG16) for {len(paths)} images ...')
    F = _concat_once(paths)
    np.savez_compressed(fp, feats=F, paths=np.array(paths))
    print('  saved concat features', F.shape); return F, {p:i for i,p in enumerate(paths)}

ALL = CACHED.copy()
ALL['prep_path']  = ALL['prep_path'].astype(str)
ALL['patient_id'] = ALL['patient_id'].astype(str)
FEATS, IDX = concat_features(ALL)
def X_for(sub): return FEATS[[IDX[p] for p in sub['prep_path'].astype(str)]]
def X_tta(sub):
    paths = sub['prep_path'].astype(str).tolist()
    views = [FEATS[[IDX[p] for p in paths]]]
    for t in ['bright_up','bright_dn','contrast','zoom']:
        views.append(_concat_once(paths, tta=t))
    return np.mean(views, axis=0)
print('FEATS:', FEATS.shape)

  extracting concat(ResNet50+VGG16) for 1464 images ...
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
  saved concat features (1464, 2560)
FEATS: (1464, 2560)
